# CAT07_PROB01: Warehouse Dual-Box Pushing

**Category 7 — Cooperative Multi-Agent Allocation (Units 2 & 4)**

**Scenario:** Two warehouse robots must coordinate to move heavy boxes to delivery zones.

**Objective:** Implement a cooperative task-allocation policy for two agents. Each task has a value; agents should complete distinct available tasks and avoid duplicate assignments.

**State space S:** Binary tuple indicating whether each task remains available (1) or is completed (0).

**Action space A:** Select one task index from `0` to `number_of_tasks - 1`. Both agents select an action each simulation step.

**Reward / transition:** Completing an available task earns its value minus movement cost. Duplicate or unavailable assignments receive a penalty. Completing two distinct tasks in one step earns a team bonus. The state updates by marking completed tasks unavailable.

**Your deliverables:** (1) implement epsilon-greedy `choose_action`; (2) implement the Q-learning update; (3) run the supplied simulation and interpret the reward curve and learned task preferences.

**Parameters:** 4 tasks; task values = [8, 6, 9, 5].


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
random.seed(42)
np.random.seed(42)
ALPHA = 0.20
GAMMA = 0.90
EPSILON = 0.15
EPISODES = 250
MAX_STEPS = 1

PROBLEM_TITLE = 'Warehouse Dual-Box Pushing'
TASK_VALUES = [8, 6, 9, 5]
MOVEMENT_COST = 0.5
DUPLICATE_PENALTY = 2.0


In [ ]:
class EnvironmentEngine:
    def __init__(self, task_values, movement_cost=0.5, duplicate_penalty=2.0):
        self.task_values = np.array(task_values, dtype=float)
        self.n_tasks = len(task_values)
        self.movement_cost = float(movement_cost)
        self.duplicate_penalty = float(duplicate_penalty)
        self.reset()

    def reset(self):
        self.remaining = np.ones(self.n_tasks, dtype=int)
        self.done = False
        return tuple(self.remaining.tolist())

    def step(self, actions):
        # actions contains one task index selected by each of the two agents
        actions = [int(a) for a in actions]
        reward = 0.0
        completed = set()
        for a in actions:
            if a < 0 or a >= self.n_tasks:
                reward -= self.duplicate_penalty
            elif self.remaining[a] and a not in completed:
                reward += self.task_values[a] - self.movement_cost
                completed.add(a)
            else:
                reward -= self.duplicate_penalty
        for a in completed:
            self.remaining[a] = 0
        self.done = not bool(np.any(self.remaining))
        # Small team bonus encourages completing distinct useful tasks together.
        if len(completed) == 2:
            reward += 1.0
        return tuple(self.remaining.tolist()), reward, self.done, {"completed": sorted(completed)}


In [ ]:
def choose_action(state, q_table, epsilon=EPSILON):
    ### YOUR CODE HERE ###
    pass


In [ ]:
def update_q_value(q_table, action, reward, next_state, alpha=ALPHA, gamma=GAMMA):
    ### YOUR CODE HERE ###
    pass


In [ ]:
env = EnvironmentEngine(TASK_VALUES, movement_cost=MOVEMENT_COST, duplicate_penalty=DUPLICATE_PENALTY)
q1 = np.zeros(env.n_tasks, dtype=float)
q2 = np.zeros(env.n_tasks, dtype=float)
episode_rewards = []
episode_steps = []

for episode in range(EPISODES):
    state = env.reset()
    total_reward = 0.0
    steps = 0
    while not env.done and steps < env.n_tasks + 2:
        a1 = choose_action(state, q1, EPSILON)
        # Agent 2 uses its own policy; a shared state helps the agents coordinate.
        a2 = choose_action(state, q2, EPSILON)
        next_state, reward, done, info = env.step([a1, a2])
        update_q_value(q1, a1, reward, next_state)
        update_q_value(q2, a2, reward, next_state)
        state = next_state
        total_reward += reward
        steps += 1
    episode_rewards.append(total_reward)
    episode_steps.append(steps)

print(f"Tasks: {env.n_tasks} | Episodes: {EPISODES}")
print(f"Mean reward (last 25 episodes): {np.mean(episode_rewards[-25:]):.2f}")
print(f"Mean steps (last 25 episodes): {np.mean(episode_steps[-25:]):.2f}")


In [ ]:
window = 10
smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(episode_rewards, alpha=0.35, label="Episode reward")
plt.plot(range(window - 1, len(episode_rewards)), smoothed, label=f"{window}-episode moving average")
plt.xlabel("Episode")
plt.ylabel("Cumulative team reward")
plt.title(PROBLEM_TITLE + " — Learning Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Learned task preferences (Agent 1):", np.round(q1, 2))
print("Learned task preferences (Agent 2):", np.round(q2, 2))
